In [1]:
# 1. Install the process mining library
!pip install pm4py -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.3 MB/s eta 0:00:0000:0100:01


In [2]:
import pm4py
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully")
print(f"PyTorch version: {torch.__version__}")

All libraries loaded successfully
PyTorch version: 2.9.0+cpu


In [3]:
# Cell 3: Load XES datasets
# Update paths to match your Kaggle input folder name exactly
PATH_DOMESTIC = "/kaggle/input/datasets/wasif8926/bpi-challenge-2020/DomesticDeclarations.xes"
PATH_INTERNATIONAL = "/kaggle/input/datasets/wasif8926/bpi-challenge-2020/InternationalDeclarations.xes"

# Load XES files
log_domestic = pm4py.read_xes(PATH_DOMESTIC)
log_international = pm4py.read_xes(PATH_INTERNATIONAL)

# Convert to dataframes
df_domestic = pm4py.convert_to_dataframe(log_domestic)
df_international = pm4py.convert_to_dataframe(log_international)

# Add source label — important for later analysis
df_domestic['dataset_source'] = 'domestic'
df_international['dataset_source'] = 'international'

# Merge both datasets
df = pd.concat([df_domestic, df_international], ignore_index=True)

print(f"Domestic records:      {len(df_domestic):,}")
print(f"International records: {len(df_international):,}")
print(f"Combined records:      {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

parsing log, completed traces ::   0%|          | 0/10500 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/6449 [00:00<?, ?it/s]

Domestic records:      56,437
International records: 72,151
Combined records:      128,588
Columns: ['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role', 'case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'dataset_source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']


In [4]:
# Cell 4: Select the 14 features from your ERP Feature Schema
# case:Permit ID is kept here for cross-entity join logic, dropped later before model input

SELECTED_COLUMNS = [
    # Control Flow
    'concept:name',
    # Resource
    'org:resource',
    'org:role',
    # Financial
    'case:Amount',
    'case:RequestedAmount',
    'case:AdjustedAmount',
    'case:OriginalAmount',
    'case:Permit RequestedBudget',
    # Entity Relationship (join key — preprocessing only)
    'case:Permit ID',
    # Entity Relationship (model inputs)
    'case:Permit BudgetNumber',
    'case:BudgetNumber',
    'case:Permit ProjectNumber',
    # Organizational Context
    'case:Permit OrganizationalEntity',
    # Temporal source
    'time:timestamp',
    # Case identifier for grouping
    'case:concept:name'  # this is your Case ID column
]

# Keep only selected columns that actually exist in the dataframe
existing_cols = [c for c in SELECTED_COLUMNS if c in df.columns]
missing_cols = [c for c in SELECTED_COLUMNS if c not in df.columns]

df = df[existing_cols].copy()

print(f"Selected columns: {len(existing_cols)}")
print(f"Missing columns:  {missing_cols}")
print(f"\nShape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

Selected columns: 15
Missing columns:  []

Shape: (128588, 15)

Data types:
concept:name                                     object
org:resource                                     object
org:role                                         object
case:Amount                                     float64
case:RequestedAmount                            float64
case:AdjustedAmount                             float64
case:OriginalAmount                             float64
case:Permit RequestedBudget                     float64
case:Permit ID                                   object
case:Permit BudgetNumber                         object
case:BudgetNumber                                object
case:Permit ProjectNumber                        object
case:Permit OrganizationalEntity                 object
time:timestamp                      datetime64[ns, UTC]
case:concept:name                                object
dtype: object


In [5]:
# Cell 5: Inspect the data before any engineering
print("=== NULL VALUES ===")
print(df.isnull().sum())

print("\n=== UNIQUE VALUES PER COLUMN ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

print("\n=== SAMPLE ROWS ===")
df.head(3)

=== NULL VALUES ===
concept:name                            0
org:resource                            0
org:role                                0
case:Amount                             0
case:RequestedAmount                56437
case:AdjustedAmount                 56437
case:OriginalAmount                 56437
case:Permit RequestedBudget         56437
case:Permit ID                      56437
case:Permit BudgetNumber            56437
case:BudgetNumber                       0
case:Permit ProjectNumber           56437
case:Permit OrganizationalEntity    56437
time:timestamp                          0
case:concept:name                       0
dtype: int64

=== UNIQUE VALUES PER COLUMN ===
concept:name: 37 unique values
org:resource: 2 unique values
org:role: 8 unique values
case:Amount: 14395 unique values
case:RequestedAmount: 6100 unique values
case:AdjustedAmount: 6101 unique values
case:OriginalAmount: 6100 unique values
case:Permit RequestedBudget: 5259 unique values
case:Permit ID

,concept:name,org:resource,org:role,case:Amount,case:RequestedAmount,case:AdjustedAmount,case:OriginalAmount,case:Permit RequestedBudget,case:Permit ID,case:Permit BudgetNumber,case:BudgetNumber,case:Permit ProjectNumber,case:Permit OrganizationalEntity,time:timestamp,case:concept:name
0,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,26.851205,NaN,NaN,NaN,NaN,NaN,NaN,budget 86566,NaN,NaN,2017-01-09 09:49:50+00:00,declaration 86791
1,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,26.851205,NaN,NaN,NaN,NaN,NaN,NaN,budget 86566,NaN,NaN,2017-01-09 11:27:48+00:00,declaration 86791
2,Request Payment,SYSTEM,UNDEFINED,26.851205,NaN,NaN,NaN,NaN,NaN,NaN,budget 86566,NaN,NaN,2017-01-10 09:34:44+00:00,declaration 86791


In [6]:
# Cell 6: Handle missing values
# 
# WHY: Neural networks cannot process NaN values.
# 
# STRATEGY:
# Financial columns → fill with 0
#   Reason: NaN means no permit was filed, so no financial value exists.
#   Zero correctly represents "no amount" rather than inventing a value.
#
# Categorical columns → fill with "UNKNOWN"  
#   Reason: NaN means no permit was linked to this declaration.
#   "UNKNOWN" becomes its own category during label encoding.
#   The model learns that UNKNOWN = domestic declaration with no permit.

financial_cols = [
    'case:Amount',
    'case:RequestedAmount',
    'case:AdjustedAmount',
    'case:OriginalAmount',
    'case:Permit RequestedBudget'
]

categorical_cols = [
    'org:resource',
    'org:role',
    'case:Permit ID',
    'case:Permit BudgetNumber',
    'case:BudgetNumber',
    'case:Permit ProjectNumber',
    'case:Permit OrganizationalEntity'
]

# Fill financial nulls with 0
df[financial_cols] = df[financial_cols].fillna(0)

# Fill categorical nulls with UNKNOWN
df[categorical_cols] = df[categorical_cols].fillna('UNKNOWN')

# Verify no nulls remain (except time:timestamp which has none)
print("=== NULL VALUES AFTER HANDLING ===")
print(df.isnull().sum())

print("\n=== VERIFICATION ===")
print(f"Total nulls remaining: {df.isnull().sum().sum()}")

print("\n=== FINANCIAL COLUMN SAMPLE (first 3 domestic rows) ===")
print(df[financial_cols].head(3))

print("\n=== CATEGORICAL COLUMN SAMPLE ===")
print(df[categorical_cols].head(3))

=== NULL VALUES AFTER HANDLING ===
concept:name                        0
org:resource                        0
org:role                            0
case:Amount                         0
case:RequestedAmount                0
case:AdjustedAmount                 0
case:OriginalAmount                 0
case:Permit RequestedBudget         0
case:Permit ID                      0
case:Permit BudgetNumber            0
case:BudgetNumber                   0
case:Permit ProjectNumber           0
case:Permit OrganizationalEntity    0
time:timestamp                      0
case:concept:name                   0
dtype: int64

=== VERIFICATION ===
Total nulls remaining: 0

=== FINANCIAL COLUMN SAMPLE (first 3 domestic rows) ===
   case:Amount  case:RequestedAmount  case:AdjustedAmount  \
0    26.851205                   0.0                  0.0   
1    26.851205                   0.0                  0.0   
2    26.851205                   0.0                  0.0   

   case:OriginalAmount  case:Perm

In [7]:
# Cell 7: Derive temporal features
#
# WHY: Raw timestamps have no process-relative meaning for neural networks.
# We derive two features that capture process rhythm:
#
# time_since_last_event: seconds between consecutive events in the same case
#   → Detects temporal anomalies (impossibly fast or reversed event sequences)
#
# case_duration: seconds from first event to current event in the same case
#   → Detects abnormally short or long process executions
#
# CRITICAL: Must sort by case ID + timestamp FIRST
#           Otherwise differences are computed in wrong order

# Step 1: Ensure timestamp is datetime with timezone
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True)

# Step 2: Sort by case ID then timestamp — this is mandatory
df = df.sort_values(
    ['case:concept:name', 'time:timestamp']
).reset_index(drop=True)

# Step 3: time_since_last_event
# .diff() computes difference between consecutive rows within each group
# .dt.total_seconds() converts timedelta to seconds
# .fillna(0) sets first event of each case to 0 (no previous event)
df['time_since_last_event'] = (
    df.groupby('case:concept:name')['time:timestamp']
    .diff()
    .dt.total_seconds()
    .fillna(0)
)

# Step 4: case_duration
# For each event: current timestamp minus first timestamp in the case
df['case_duration'] = (
    df.groupby('case:concept:name')['time:timestamp']
    .transform(lambda x: (x - x.min()).dt.total_seconds())
)

# Verification
print("=== TEMPORAL FEATURES CREATED ===")
print(df[['case:concept:name', 
          'concept:name',
          'time:timestamp', 
          'time_since_last_event', 
          'case_duration']].head(10).to_string())

print("\n=== RANGE CHECK ===")
print(f"time_since_last_event:")
print(f"  Min: {df['time_since_last_event'].min():,.0f} seconds")
print(f"  Max: {df['time_since_last_event'].max():,.0f} seconds")
print(f"  Mean: {df['time_since_last_event'].mean():,.0f} seconds "
      f"({df['time_since_last_event'].mean()/3600:.1f} hours)")

print(f"\ncase_duration:")
print(f"  Min: {df['case_duration'].min():,.0f} seconds")
print(f"  Max: {df['case_duration'].max():,.0f} seconds")
print(f"  Mean: {df['case_duration'].mean()/86400:.1f} days")

print(f"\n=== NULL CHECK ===")
print(f"time_since_last_event nulls: {df['time_since_last_event'].isnull().sum()}")
print(f"case_duration nulls:         {df['case_duration'].isnull().sum()}")

print(f"\n=== SHAPE ===")
print(f"DataFrame shape: {df.shape}")
print(f"New columns added: time_since_last_event, case_duration")

=== TEMPORAL FEATURES CREATED ===
    case:concept:name                              concept:name            time:timestamp  time_since_last_event  case_duration
0  declaration 100000         Declaration SUBMITTED by EMPLOYEE 2018-01-30 09:20:07+00:00                    0.0            0.0
1  declaration 100000    Declaration APPROVED by ADMINISTRATION 2018-02-07 09:58:46+00:00               693519.0       693519.0
2  declaration 100000  Declaration FINAL_APPROVED by SUPERVISOR 2018-02-08 10:59:05+00:00                90019.0       783538.0
3  declaration 100000                           Request Payment 2018-02-09 12:42:49+00:00                92624.0       876162.0
4  declaration 100000                           Payment Handled 2018-02-12 17:31:20+00:00               276511.0      1152673.0
5  declaration 100005         Declaration SUBMITTED by EMPLOYEE 2018-01-30 09:38:54+00:00                    0.0            0.0
6  declaration 100005    Declaration APPROVED by ADMINISTRATION 2018-0

In [8]:
# Cell 8: Label encode all categorical features
#
# WHY: Neural networks require numerical inputs.
# Text strings must be converted to integers.
#
# METHOD: LabelEncoder maps each unique string to a unique integer.
# Encoders are saved in a dictionary so we can:
#   1. Decode predictions back to activity names during evaluation
#   2. Reference specific activity integers during anomaly injection
#
# NOTE: Fitting on full dataset is acceptable for label encoding.
# We are only building a vocabulary, not learning statistical parameters.
# Numerical scaling (Cell 9) will be fit on train only.

encoders = {}

categorical_to_encode = [
    'concept:name',                    # sequence target — 37 unique activities
    'org:resource',                    # 2 unique values (STAFF MEMBER, SYSTEM)
    'org:role',                        # 8 unique roles
    'case:Permit BudgetNumber',        # 207 unique + UNKNOWN
    'case:BudgetNumber',               # 720 unique budget codes
    'case:Permit ProjectNumber',       # 825 unique + UNKNOWN
    'case:Permit OrganizationalEntity' # 27 unique + UNKNOWN
]

print("=== LABEL ENCODING ===\n")
for col in categorical_to_encode:
    le = LabelEncoder()
    encoded_col = col + '_encoded'
    df[encoded_col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}")
    print(f"  Unique classes: {len(le.classes_)}")
    print(f"  Sample mapping:")
    # Show first 3 mappings
    for i, cls in enumerate(le.classes_[:3]):
        print(f"    '{cls}' → {i}")
    print()

print(f"Total encoders saved: {len(encoders)}")

# Show the concept:name mapping in full — important for anomaly injection
print("\n=== FULL ACTIVITY MAPPING (concept:name) ===")
print("(You need these integers for anomaly injection later)\n")
for i, activity in enumerate(encoders['concept:name'].classes_):
    print(f"  {i:2d} → {activity}")

=== LABEL ENCODING ===

concept:name
  Unique classes: 37
  Sample mapping:
    'Declaration APPROVED by ADMINISTRATION' → 0
    'Declaration APPROVED by BUDGET OWNER' → 1
    'Declaration APPROVED by PRE_APPROVER' → 2

org:resource
  Unique classes: 2
  Sample mapping:
    'STAFF MEMBER' → 0
    'SYSTEM' → 1

org:role
  Unique classes: 8
  Sample mapping:
    'ADMINISTRATION' → 0
    'BUDGET OWNER' → 1
    'DIRECTOR' → 2

case:Permit BudgetNumber
  Unique classes: 208
  Sample mapping:
    'UNKNOWN' → 0
    'budget 1021' → 1
    'budget 10229' → 2

case:BudgetNumber
  Unique classes: 720
  Sample mapping:
    'UNKNOWN' → 0
    'budget 143466' → 1
    'budget 143471' → 2

case:Permit ProjectNumber
  Unique classes: 825
  Sample mapping:
    'UNKNOWN' → 0
    'project 10042' → 1
    'project 10068' → 2

case:Permit OrganizationalEntity
  Unique classes: 28
  Sample mapping:
    'UNKNOWN' → 0
    'organizational unit 65454' → 1
    'organizational unit 65455' → 2

Total encoders saved: 7

In [9]:
# Cell 9: Train/Test Split by Case ID
#
# WHY CASE-LEVEL SPLIT:
# Splitting by row would put parts of the same trace in both train and test.
# The model would partially memorize test traces during training.
# Case-level split ensures the model never sees any part of a test case
# during training. This is the only valid evaluation strategy.
#
# 80% train → model learns normal ERP behavior from clean traces
# 20% test  → anomalies will be injected here later

case_ids = df['case:concept:name'].unique()
print(f"Total unique cases: {len(case_ids):,}")

train_cases, test_cases = train_test_split(
    case_ids,
    test_size=0.2,
    random_state=42  # fixed seed for reproducibility
)

df_train = df[df['case:concept:name'].isin(train_cases)].copy()
df_test  = df[df['case:concept:name'].isin(test_cases)].copy()

print(f"\n=== SPLIT RESULTS ===")
print(f"Train cases:  {len(train_cases):,}  "
      f"({len(train_cases)/len(case_ids)*100:.1f}%)")
print(f"Test cases:   {len(test_cases):,}   "
      f"({len(test_cases)/len(case_ids)*100:.1f}%)")
print(f"\nTrain events: {len(df_train):,}")
print(f"Test events:  {len(df_test):,}")
print(f"Total events: {len(df_train) + len(df_test):,} "
      f"(should match {len(df):,})")

print(f"\n=== SANITY CHECK ===")
# Confirm no case appears in both sets
overlap = set(train_cases) & set(test_cases)
print(f"Cases in both train and test: {len(overlap)} (must be 0)")

Total unique cases: 16,949

=== SPLIT RESULTS ===
Train cases:  13,559  (80.0%)
Test cases:   3,390   (20.0%)

Train events: 102,960
Test events:  25,628
Total events: 128,588 (should match 128,588)

=== SANITY CHECK ===
Cases in both train and test: 0 (must be 0)


In [10]:
# Cell 10: Normalize numerical features
#
# WHY FIT ON TRAIN ONLY:
# MinMaxScaler learns min and max values from the data.
# If we fit on the full dataset, test set statistics influence the scaler.
# This is data leakage — the model indirectly "sees" test data during training.
#
# CORRECT APPROACH:
# fit on train → learn min/max from training cases only
# transform both train and test → apply same scaling formula

scalers = {}

numerical_to_scale = [
    'case:Amount',
    'case:RequestedAmount',
    'case:AdjustedAmount',
    'case:OriginalAmount',
    'case:Permit RequestedBudget',
    'time_since_last_event',
    'case_duration'
]

print("=== FITTING SCALERS ON TRAIN SET ONLY ===\n")
for col in numerical_to_scale:
    scaler = MinMaxScaler()
    
    # Fit ONLY on training data
    scaler.fit(df_train[[col]])
    
    # Transform both train and test using same scaler
    df_train[col + '_scaled'] = scaler.transform(df_train[[col]])
    df_test[col + '_scaled']  = scaler.transform(df_test[[col]])
    
    scalers[col] = scaler
    print(f"{col}")
    print(f"  Train min: {scaler.data_min_[0]:.4f} → 0.0")
    print(f"  Train max: {scaler.data_max_[0]:.4f} → 1.0")
    print()

print(f"Scalers saved: {len(scalers)}")

print("\n=== VERIFICATION SAMPLE (train) ===")
sample_cols = [c + '_scaled' for c in numerical_to_scale]
print(df_train[sample_cols].head(3).to_string())

print("\n=== NULL CHECK ===")
print(f"Train nulls: {df_train[sample_cols].isnull().sum().sum()}")
print(f"Test nulls:  {df_test[sample_cols].isnull().sum().sum()}")

=== FITTING SCALERS ON TRAIN SET ONLY ===

case:Amount
  Train min: 0.0000 → 0.0
  Train max: 10765.9355 → 1.0

case:RequestedAmount
  Train min: 0.0000 → 0.0
  Train max: 10765.9355 → 1.0

case:AdjustedAmount
  Train min: 0.0000 → 0.0
  Train max: 10765.9355 → 1.0

case:OriginalAmount
  Train min: 0.0000 → 0.0
  Train max: 10765.9355 → 1.0

case:Permit RequestedBudget
  Train min: 0.0000 → 0.0
  Train max: 11112.3851 → 1.0

time_since_last_event
  Train min: 0.0000 → 0.0
  Train max: 57047277.0000 → 1.0

case_duration
  Train min: 0.0000 → 0.0
  Train max: 64108800.0000 → 1.0

Scalers saved: 7

=== VERIFICATION SAMPLE (train) ===
   case:Amount_scaled  case:RequestedAmount_scaled  case:AdjustedAmount_scaled  case:OriginalAmount_scaled  case:Permit RequestedBudget_scaled  time_since_last_event_scaled  case_duration_scaled
5            0.003263                          0.0                         0.0                         0.0                                 0.0                  0.0000

In [11]:
# Cell 11: Define final model input features
#
# These 14 features form the event vector fed to LSTM/GRU/Transformer.
# case:Permit ID → dropped (join key only, preprocessing done)
# time:timestamp → dropped (already derived into temporal features)
# case:concept:name → kept as grouping key only, not model input

MODEL_FEATURES = [
    # Control Flow (label encoded)
    'concept:name_encoded',           # feature 1
    # Resource (label encoded)
    'org:resource_encoded',           # feature 2
    'org:role_encoded',               # feature 3
    # Financial (scaled)
    'case:Amount_scaled',             # feature 4
    'case:RequestedAmount_scaled',    # feature 5
    'case:AdjustedAmount_scaled',     # feature 6
    'case:OriginalAmount_scaled',     # feature 7
    'case:Permit RequestedBudget_scaled', # feature 8
    # Entity Relationship (label encoded)
    'case:Permit BudgetNumber_encoded',   # feature 9
    'case:BudgetNumber_encoded',          # feature 10
    'case:Permit ProjectNumber_encoded',  # feature 11
    # Organizational Context (label encoded)
    'case:Permit OrganizationalEntity_encoded', # feature 12
    # Temporal (scaled)
    'time_since_last_event_scaled',   # feature 13
    'case_duration_scaled'            # feature 14
]

TARGET_COL   = 'concept:name_encoded'  # what the model predicts
CASE_ID_COL  = 'case:concept:name'     # grouping key

print(f"Model input features: {len(MODEL_FEATURES)}")
print(f"Target column: {TARGET_COL}")
print(f"\nFeature breakdown:")
print(f"  Label encoded categoricals: 7")
print(f"  MinMax scaled numericals:   7")
print(f"  Total:                     14")

print(f"\nFull feature list:")
for i, f in enumerate(MODEL_FEATURES, 1):
    print(f"  {i:2d}. {f}")

# Verify all features exist in both train and test
missing_train = [f for f in MODEL_FEATURES if f not in df_train.columns]
missing_test  = [f for f in MODEL_FEATURES if f not in df_test.columns]
print(f"\nMissing from train: {missing_train}")
print(f"Missing from test:  {missing_test}")
print(f"\nAll features present: "
      f"{'YES' if not missing_train and not missing_test else 'NO - CHECK ABOVE'}")

Model input features: 14
Target column: concept:name_encoded

Feature breakdown:
  Label encoded categoricals: 7
  MinMax scaled numericals:   7
  Total:                     14

Full feature list:
   1. concept:name_encoded
   2. org:resource_encoded
   3. org:role_encoded
   4. case:Amount_scaled
   5. case:RequestedAmount_scaled
   6. case:AdjustedAmount_scaled
   7. case:OriginalAmount_scaled
   8. case:Permit RequestedBudget_scaled
   9. case:Permit BudgetNumber_encoded
  10. case:BudgetNumber_encoded
  11. case:Permit ProjectNumber_encoded
  12. case:Permit OrganizationalEntity_encoded
  13. time_since_last_event_scaled
  14. case_duration_scaled

Missing from train: []
Missing from test:  []

All features present: YES


In [12]:
# Cell 12: Create sequences per case ID
#
# WHY: Neural networks consume fixed-size tensors.
# Each case becomes one sequence (matrix of event vectors).
# Variable-length sequences are zero-padded to a fixed maximum length.
#
# OUTPUT:
# sequences: list of arrays, each shape (max_len, 14)
# targets:   list of integers, the last activity in each sequence
# case_ids:  list of case identifiers for later anomaly injection

def create_sequences(dataframe, feature_cols, target_col, 
                     case_col, min_length=2):
    """
    Groups events by case ID and creates (sequence, target) pairs.
    
    Parameters:
        dataframe:    input dataframe (train or test)
        feature_cols: list of feature column names
        target_col:   column to predict (concept:name_encoded)
        case_col:     case grouping column
        min_length:   minimum events per case (skip shorter cases)
    
    Returns:
        sequences:  list of numpy arrays, each shape (n_events, n_features)
        targets:    list of integers (last activity integer)
        case_ids:   list of case ID strings
    """
    sequences  = []
    targets    = []
    case_ids   = []
    skipped    = 0

    for case_id, group in dataframe.groupby(case_col):
        # Sort by timestamp within case
        group = group.sort_values('time:timestamp')

        if len(group) < min_length:
            skipped += 1
            continue

        feature_seq = group[feature_cols].values.astype(np.float32)
        target_val  = group[target_col].values[-1]  # last activity as target

        sequences.append(feature_seq)
        targets.append(int(target_val))
        case_ids.append(case_id)

    return sequences, targets, case_ids


# Create sequences for train and test separately
train_sequences, train_targets, train_case_ids = create_sequences(
    df_train, MODEL_FEATURES, TARGET_COL, CASE_ID_COL
)

test_sequences, test_targets, test_case_ids = create_sequences(
    df_test, MODEL_FEATURES, TARGET_COL, CASE_ID_COL
)

print(f"=== SEQUENCE CREATION RESULTS ===")
print(f"Train sequences: {len(train_sequences):,}")
print(f"Test sequences:  {len(test_sequences):,}")

print(f"\n=== SEQUENCE LENGTH STATISTICS ===")
train_lengths = [len(s) for s in train_sequences]
test_lengths  = [len(s) for s in test_sequences]

print(f"Train — min: {min(train_lengths)}, "
      f"max: {max(train_lengths)}, "
      f"mean: {np.mean(train_lengths):.1f}")
print(f"Test  — min: {min(test_lengths)}, "
      f"max: {max(test_lengths)}, "
      f"mean: {np.mean(test_lengths):.1f}")

print(f"\n=== EXAMPLE SEQUENCE ===")
print(f"Case: {train_case_ids[0]}")
print(f"Sequence shape: {train_sequences[0].shape}")
print(f"  → {train_sequences[0].shape[0]} events × "
      f"{train_sequences[0].shape[1]} features")
print(f"Target activity integer: {train_targets[0]}")
print(f"Target activity name: "
      f"{encoders['concept:name'].classes_[train_targets[0]]}")

print(f"\n=== FEATURE VECTOR OF FIRST EVENT IN FIRST SEQUENCE ===")
for i, fname in enumerate(MODEL_FEATURES):
    print(f"  {fname:<45} {train_sequences[0][0][i]:.6f}")

=== SEQUENCE CREATION RESULTS ===
Train sequences: 13,448
Test sequences:  3,367

=== SEQUENCE LENGTH STATISTICS ===
Train — min: 2, max: 27, mean: 7.6
Test  — min: 3, max: 27, mean: 7.6

=== EXAMPLE SEQUENCE ===
Case: declaration 100005
Sequence shape: (5, 14)
  → 5 events × 14 features
Target activity integer: 19
Target activity name: Payment Handled

=== FEATURE VECTOR OF FIRST EVENT IN FIRST SEQUENCE ===
  concept:name_encoded                          17.000000
  org:resource_encoded                          0.000000
  org:role_encoded                              3.000000
  case:Amount_scaled                            0.003263
  case:RequestedAmount_scaled                   0.000000
  case:AdjustedAmount_scaled                    0.000000
  case:OriginalAmount_scaled                    0.000000
  case:Permit RequestedBudget_scaled            0.000000
  case:Permit BudgetNumber_encoded              0.000000
  case:BudgetNumber_encoded                     719.000000
  case:Permit P

In [13]:
# Cell 13: Padding + PyTorch Dataset and DataLoader
#
# WHY PADDING:
# PyTorch requires fixed-size tensors within a batch.
# All sequences padded to MAX_SEQ_LEN (27) with zero vectors.
# Zero padding is safe — model learns padded rows carry no information.
#
# DATASET:
# ERPSequenceDataset wraps sequences and targets into PyTorch format.
# DataLoader handles batching and shuffling during training.

MAX_SEQ_LEN  = max(max(len(s) for s in train_sequences),
                   max(len(s) for s in test_sequences))
N_FEATURES   = len(MODEL_FEATURES)

print(f"Maximum sequence length: {MAX_SEQ_LEN}")
print(f"Number of features:      {N_FEATURES}")

def pad_sequences(sequences, max_len, n_features):
    """
    Pads all sequences to max_len with zero vectors.
    
    Input:  list of arrays, each shape (seq_len, n_features)
    Output: single array of shape (n_sequences, max_len, n_features)
    """
    padded = np.zeros(
        (len(sequences), max_len, n_features), 
        dtype=np.float32
    )
    for i, seq in enumerate(sequences):
        length = len(seq)
        padded[i, :length, :] = seq
    return padded


# Pad both sets
train_padded = pad_sequences(train_sequences, MAX_SEQ_LEN, N_FEATURES)
test_padded  = pad_sequences(test_sequences,  MAX_SEQ_LEN, N_FEATURES)

print(f"\n=== PADDED ARRAY SHAPES ===")
print(f"Train: {train_padded.shape}")
print(f"  → {train_padded.shape[0]:,} sequences × "
      f"{train_padded.shape[1]} timesteps × "
      f"{train_padded.shape[2]} features")
print(f"Test:  {test_padded.shape}")


# PyTorch Dataset
class ERPSequenceDataset(Dataset):
    """
    Wraps padded sequences and integer targets into PyTorch Dataset format.
    sequences: tensor of shape (n_samples, max_len, n_features)
    targets:   tensor of shape (n_samples,) — integer activity labels
    """
    def __init__(self, sequences, targets):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.targets   = torch.tensor(targets,   dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]


# Create datasets
train_dataset = ERPSequenceDataset(train_padded, train_targets)
test_dataset  = ERPSequenceDataset(test_padded,  test_targets)

# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True        # shuffle train for better gradient updates
)
test_loader = DataLoader(
    test_dataset,  
    batch_size=BATCH_SIZE, 
    shuffle=False       # never shuffle test — order must be preserved
)

print(f"\n=== DATALOADER RESULTS ===")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Train batches:  {len(train_loader)}")
print(f"Test batches:   {len(test_loader)}")

# Inspect one batch to confirm shapes
sample_sequences, sample_targets = next(iter(train_loader))
print(f"\n=== SINGLE BATCH SHAPE CHECK ===")
print(f"Sequences batch: {sample_sequences.shape}")
print(f"  → {sample_sequences.shape[0]} cases × "
      f"{sample_sequences.shape[1]} timesteps × "
      f"{sample_sequences.shape[2]} features")
print(f"Targets batch:   {sample_targets.shape}")
print(f"  → {sample_targets.shape[0]} target labels")
print(f"\nSample target integers: {sample_targets[:8].tolist()}")
print(f"Sample target names:")
for t in sample_targets[:8].tolist():
    print(f"  {t:2d} → {encoders['concept:name'].classes_[t]}")


Maximum sequence length: 27
Number of features:      14

=== PADDED ARRAY SHAPES ===
Train: (13448, 27, 14)
  → 13,448 sequences × 27 timesteps × 14 features
Test:  (3367, 27, 14)

=== DATALOADER RESULTS ===
Batch size:     32
Train batches:  421
Test batches:   106

=== SINGLE BATCH SHAPE CHECK ===
Sequences batch: torch.Size([32, 27, 14])
  → 32 cases × 27 timesteps × 14 features
Targets batch:   torch.Size([32])
  → 32 target labels

Sample target integers: [18, 19, 19, 19, 19, 19, 19, 19]
Sample target names:
  18 → End trip
  19 → Payment Handled
  19 → Payment Handled
  19 → Payment Handled
  19 → Payment Handled
  19 → Payment Handled
  19 → Payment Handled
  19 → Payment Handled


In [14]:
# Cell 14: LSTM Model Definition
#
# ARCHITECTURE:
# 2-layer LSTM with hidden size 64
# Dropout 0.3 between layers to prevent overfitting
# Final linear layer maps hidden state to 37 activity classes
#
# INPUT:  (batch_size, seq_len=27, input_size=14)
# OUTPUT: (batch_size, num_classes=37)

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, 
                 num_classes, dropout=0.3):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        
        # LSTM layers
        # batch_first=True means input shape is (batch, seq, features)
        # dropout applies between LSTM layers (not after last layer)
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        
        # Dropout before final layer
        self.dropout = nn.Dropout(dropout)
        
        # Final classification layer
        # Maps hidden state → probability over activity classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        
        # LSTM forward pass
        # lstm_out shape: (batch_size, seq_len, hidden_size)
        # hidden, cell: internal states (not used here)
        lstm_out, _ = self.lstm(x)
        
        # Take ONLY the last timestep output
        # Shape: (batch_size, hidden_size)
        last_timestep = lstm_out[:, -1, :]
        
        # Apply dropout
        out = self.dropout(last_timestep)
        
        # Linear layer → class logits
        # Shape: (batch_size, num_classes)
        out = self.fc(out)
        
        return out


# Model configuration
INPUT_SIZE   = len(MODEL_FEATURES)   # 14
HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
NUM_CLASSES  = len(encoders['concept:name'].classes_)  # 37
DROPOUT      = 0.3

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model_lstm = LSTMModel(
    input_size  = INPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    dropout     = DROPOUT
).to(device)

# Print architecture summary
print(f"\n=== LSTM ARCHITECTURE ===")
print(model_lstm)

# Count parameters
total_params     = sum(p.numel() for p in model_lstm.parameters())
trainable_params = sum(p.numel() for p in model_lstm.parameters() 
                       if p.requires_grad)
print(f"\n=== PARAMETER COUNT ===")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Confirm forward pass works with one batch
print(f"\n=== FORWARD PASS TEST ===")
model_lstm.eval()
with torch.no_grad():
    test_input  = sample_sequences.to(device)
    test_output = model_lstm(test_input)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"  → {test_output.shape[0]} cases × "
      f"{test_output.shape[1]} class probabilities")
print(f"\nForward pass successful: "
      f"{'YES' if test_output.shape == (32, 37) else 'NO'}")

Device: cpu

=== LSTM ARCHITECTURE ===
LSTMModel(
  (lstm): LSTM(14, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=37, bias=True)
)

=== PARAMETER COUNT ===
Total parameters:     56,165
Trainable parameters: 56,165

=== FORWARD PASS TEST ===
Input shape:  torch.Size([32, 27, 14])
Output shape: torch.Size([32, 37])
  → 32 cases × 37 class probabilities

Forward pass successful: YES


In [15]:
# Cell 15: Training Loop
#
# LOSS: CrossEntropyLoss
#   Measures prediction error over 37 activity classes.
#   Expects raw logits (not softmax) — PyTorch applies softmax internally.
#
# OPTIMIZER: Adam, lr=0.001
#   Adaptive learning rate optimizer. Standard for LSTM training.
#
# EPOCHS: 15
#   Enough for prototype convergence. Full implementation will use more.

# Switch to GPU if available
# If you enabled GPU in Kaggle session options, this will show 'cuda'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model_lstm = model_lstm.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

EPOCHS = 15

# Store history for plotting
history = {
    'train_loss': [],
    'train_acc':  []
}

print(f"\n=== TRAINING LSTM ===")
print(f"{'Epoch':<8} {'Loss':<12} {'Accuracy':<12} {'Status'}")
print("-" * 45)

for epoch in range(1, EPOCHS + 1):
    model_lstm.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for sequences, labels in train_loader:
        sequences = sequences.to(device)
        labels    = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model_lstm(sequences)

        # Loss calculation
        loss = criterion(outputs, labels)

        # Backward pass + parameter update
        loss.backward()
        
        # Gradient clipping — prevents exploding gradients in LSTM
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1.0)
        
        optimizer.step()

        # Track metrics
        total_loss += loss.item()
        _, predicted = outputs.max(dim=1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    # Epoch metrics
    avg_loss  = total_loss / len(train_loader)
    train_acc = 100.0 * correct / total

    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)

    # Status indicator
    if epoch == 1:
        status = "← starting point"
    elif avg_loss < history['train_loss'][-2]:
        status = "↓ loss improving"
    else:
        status = "→ plateauing"

    print(f"{epoch:<8} {avg_loss:<12.4f} {train_acc:<11.2f}% {status}")

print("-" * 45)
print(f"\nFinal training loss:     {history['train_loss'][-1]:.4f}")
print(f"Final training accuracy: {history['train_acc'][-1]:.2f}%")
print(f"\nBaseline (random guessing on 37 classes): "
      f"{100/NUM_CLASSES:.2f}%")
print(f"Improvement over random: "
      f"{history['train_acc'][-1] - 100/NUM_CLASSES:.2f} percentage points")

Training on: cpu

=== TRAINING LSTM ===
Epoch    Loss         Accuracy     Status
---------------------------------------------
1        0.4878       91.55      % ← starting point
2        0.3342       93.24      % ↓ loss improving
3        0.3290       93.24      % ↓ loss improving
4        0.3309       93.24      % → plateauing
5        0.3301       93.24      % ↓ loss improving
6        0.3149       93.24      % ↓ loss improving
7        0.2832       94.18      % ↓ loss improving
8        0.2753       94.43      % ↓ loss improving
9        0.2756       94.43      % → plateauing
10       0.2731       94.45      % ↓ loss improving
11       0.2742       94.46      % → plateauing
12       0.2579       94.37      % ↓ loss improving
13       0.2674       94.37      % → plateauing
14       0.2464       94.10      % ↓ loss improving
15       0.2103       94.79      % ↓ loss improving
---------------------------------------------

Final training loss:     0.2103
Final training accuracy: 94.7

In [16]:
# Cell 16: Evaluation on Test Set
#
# MODEL.EVAL(): disables dropout for consistent predictions
# TORCH.NO_GRAD(): skips gradient computation (faster, less memory)
#
# METRICS:
# Accuracy — what % of final activities were predicted correctly
# F1 weighted — accounts for class imbalance (important given Payment Handled dominance)
# Per-class report — shows which activities the model handles well vs poorly

from sklearn.metrics import (classification_report, 
                              f1_score, 
                              confusion_matrix,
                              accuracy_score)

model_lstm.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs   = model_lstm(sequences)
        _, predicted = outputs.max(dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Core metrics
accuracy = accuracy_score(all_labels, all_preds) * 100
f1_w     = f1_score(all_labels, all_preds, average='weighted', 
                    zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average='macro',    
                    zero_division=0)

print(f"=== LSTM BASELINE RESULTS ===\n")
print(f"Test Accuracy:       {accuracy:.2f}%")
print(f"F1 Weighted:         {f1_w:.4f}")
print(f"F1 Macro:            {f1_macro:.4f}")
print(f"Random baseline:     {100/NUM_CLASSES:.2f}%")
print(f"\nNote: High accuracy likely reflects Payment Handled dominance.")
print(f"F1 Macro is the honest metric — it treats all 37 classes equally.")

# What the model actually predicted
print(f"\n=== PREDICTION DISTRIBUTION ===")
unique_preds, counts = np.unique(all_preds, return_counts=True)
print(f"Unique activities predicted: {len(unique_preds)} out of {NUM_CLASSES}")
print(f"\nTop 5 predicted activities:")
top5_idx = np.argsort(counts)[::-1][:5]
for idx in top5_idx:
    activity_name = encoders['concept:name'].classes_[unique_preds[idx]]
    print(f"  {counts[idx]:4d} times → {activity_name}")

# Per-class breakdown for activities that actually appear in test
print(f"\n=== PER-CLASS PERFORMANCE ===")
present_classes = np.unique(all_labels)
target_names    = [encoders['concept:name'].classes_[i] 
                   for i in present_classes]
report = classification_report(
    all_labels, all_preds,
    labels       = present_classes,
    target_names = target_names,
    zero_division = 0
)
print(report)

=== LSTM BASELINE RESULTS ===

Test Accuracy:       95.19%
F1 Weighted:         0.9365
F1 Macro:            0.2677
Random baseline:     2.70%

Note: High accuracy likely reflects Payment Handled dominance.
F1 Macro is the honest metric — it treats all 37 classes equally.

=== PREDICTION DISTRIBUTION ===
Unique activities predicted: 3 out of 37

Top 5 predicted activities:
  3306 times → Payment Handled
    41 times → Declaration REJECTED by EMPLOYEE
    20 times → End trip

=== PER-CLASS PERFORMANCE ===
                                  precision    recall  f1-score   support

Declaration REJECTED by EMPLOYEE       0.95      0.50      0.66        78
 Declaration REJECTED by MISSING       0.00      0.00      0.00         4
   Declaration SAVED by EMPLOYEE       0.00      0.00      0.00        12
                        End trip       0.85      0.14      0.24       120
                 Payment Handled       0.95      1.00      0.98      3150
      Permit REJECTED by MISSING       0.00   

In [17]:
# Cell 17: Rebuild sequences for step-by-step next-activity prediction
#
# WHY: Predicting only the final activity causes class imbalance.
# Step-by-step prediction gives richer training signal and enables
# per-event anomaly scoring at inference time.
#
# For a trace of length N, create N-1 training pairs:
#   prefix of length 1 → predict event 2
#   prefix of length 2 → predict event 3
#   ...
#   prefix of length N-1 → predict event N

def create_prefix_sequences(dataframe, feature_cols, 
                             target_col, case_col, min_length=2):
    """
    Creates (prefix_sequence, next_activity) pairs from each trace.
    Each trace of length N produces N-1 training pairs.
    """
    sequences = []
    targets   = []
    case_ids  = []

    for case_id, group in dataframe.groupby(case_col):
        group = group.sort_values('time:timestamp')
        
        if len(group) < min_length:
            continue

        feature_matrix = group[feature_cols].values.astype(np.float32)
        target_vector  = group[target_col].values.astype(np.int64)

        # Create one pair per prefix
        for i in range(1, len(feature_matrix)):
            prefix = feature_matrix[:i]        # events 0 to i-1
            target = int(target_vector[i])     # event i is the target
            sequences.append(prefix)
            targets.append(target)
            case_ids.append(case_id)

    return sequences, targets, case_ids


train_seq_v2, train_tgt_v2, train_cid_v2 = create_prefix_sequences(
    df_train, MODEL_FEATURES, TARGET_COL, CASE_ID_COL
)
test_seq_v2, test_tgt_v2, test_cid_v2 = create_prefix_sequences(
    df_test, MODEL_FEATURES, TARGET_COL, CASE_ID_COL
)

print(f"=== PREFIX SEQUENCE RESULTS ===")
print(f"Train sequences: {len(train_seq_v2):,}  "
      f"(was {len(train_sequences):,})")
print(f"Test sequences:  {len(test_seq_v2):,}  "
      f"(was {len(test_sequences):,})")

print(f"\n=== TARGET DISTRIBUTION (train, top 10) ===")
from collections import Counter
target_counts = Counter(train_tgt_v2)
total_targets = len(train_tgt_v2)
for activity_int, count in target_counts.most_common(10):
    activity_name = encoders['concept:name'].classes_[activity_int]
    pct = 100 * count / total_targets
    print(f"  {pct:5.1f}%  ({count:5d})  → {activity_name}")

=== PREFIX SEQUENCE RESULTS ===
Train sequences: 89,401  (was 13,448)
Test sequences:  22,238  (was 3,367)

=== TARGET DISTRIBUTION (train, top 10) ===
   14.5%  (12975)  → Payment Handled
   14.5%  (12968)  → Request Payment
   14.5%  (12939)  → Declaration FINAL_APPROVED by SUPERVISOR
   11.8%  (10560)  → Declaration APPROVED by ADMINISTRATION
    8.0%  ( 7109)  → Declaration SUBMITTED by EMPLOYEE
    5.8%  ( 5169)  → End trip
    5.1%  ( 4597)  → Start trip
    4.8%  ( 4319)  → Permit FINAL_APPROVED by SUPERVISOR
    4.3%  ( 3849)  → Permit APPROVED by ADMINISTRATION
    4.2%  ( 3744)  → Declaration APPROVED by BUDGET OWNER


In [18]:
# Cell 18: Rebuild DataLoader with prefix sequences
#
# Same padding and DataLoader logic as Cell 13
# but now using the prefix sequences from Cell 17
# Train: 89,401 sequences | Test: 22,238 sequences

MAX_SEQ_LEN_V2 = max(
    max(len(s) for s in train_seq_v2),
    max(len(s) for s in test_seq_v2)
)

print(f"Max sequence length: {MAX_SEQ_LEN_V2}")
print(f"Number of features:  {N_FEATURES}")

# Pad sequences
train_padded_v2 = pad_sequences(train_seq_v2, MAX_SEQ_LEN_V2, N_FEATURES)
test_padded_v2  = pad_sequences(test_seq_v2,  MAX_SEQ_LEN_V2, N_FEATURES)

print(f"\n=== PADDED SHAPES ===")
print(f"Train: {train_padded_v2.shape}")
print(f"  → {train_padded_v2.shape[0]:,} sequences × "
      f"{train_padded_v2.shape[1]} timesteps × "
      f"{train_padded_v2.shape[2]} features")
print(f"Test:  {test_padded_v2.shape}")

# Create datasets and loaders
train_dataset_v2 = ERPSequenceDataset(train_padded_v2, train_tgt_v2)
test_dataset_v2  = ERPSequenceDataset(test_padded_v2,  test_tgt_v2)

train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=64,   # larger batch = faster training on more data
    shuffle=True
)
test_loader_v2 = DataLoader(
    test_dataset_v2,
    batch_size=64,
    shuffle=False
)

print(f"\n=== DATALOADER ===")
print(f"Batch size:     64")
print(f"Train batches:  {len(train_loader_v2)}")
print(f"Test batches:   {len(test_loader_v2)}")

# Verify batch shape
sample_seq_v2, sample_tgt_v2 = next(iter(train_loader_v2))
print(f"\n=== BATCH SHAPE CHECK ===")
print(f"Sequences: {sample_seq_v2.shape}")
print(f"Targets:   {sample_tgt_v2.shape}")
print(f"\nSample targets (first 8):")
for t in sample_tgt_v2[:8].tolist():
    print(f"  {t:2d} → {encoders['concept:name'].classes_[t]}")

Max sequence length: 26
Number of features:  14

=== PADDED SHAPES ===
Train: (89401, 26, 14)
  → 89,401 sequences × 26 timesteps × 14 features
Test:  (22238, 26, 14)

=== DATALOADER ===
Batch size:     64
Train batches:  1397
Test batches:   348

=== BATCH SHAPE CHECK ===
Sequences: torch.Size([64, 26, 14])
Targets:   torch.Size([64])

Sample targets (first 8):
   2 → Declaration APPROVED by PRE_APPROVER
  25 → Permit FINAL_APPROVED by SUPERVISOR
  34 → Request Payment
  36 → Start trip
  22 → Permit APPROVED by PRE_APPROVER
  25 → Permit FINAL_APPROVED by SUPERVISOR
  12 → Declaration REJECTED by EMPLOYEE
  19 → Payment Handled


In [19]:
# Cell 19: Retrain LSTM with prefix sequences
#
# Fresh model initialization — same architecture as Cell 14
# Training on 89,401 prefix sequences instead of 13,448 full traces
# Now predicts next activity at every step, not just final activity

# Reinitialize fresh model
model_lstm_v2 = LSTMModel(
    input_size  = INPUT_SIZE,
    hidden_size = 64,
    num_layers  = 2,
    num_classes = NUM_CLASSES,
    dropout     = 0.3
).to(device)

criterion_v2 = nn.CrossEntropyLoss()
optimizer_v2 = torch.optim.Adam(model_lstm_v2.parameters(), lr=0.001)

EPOCHS_V2 = 15

history_v2 = {'train_loss': [], 'train_acc': []}

print(f"Training on: {device}")
print(f"Training samples: {len(train_dataset_v2):,}")
print(f"\n=== TRAINING LSTM v2 (Prefix Sequences) ===")
print(f"{'Epoch':<8} {'Loss':<12} {'Accuracy':<12} {'Status'}")
print("-" * 50)

for epoch in range(1, EPOCHS_V2 + 1):
    model_lstm_v2.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for sequences, labels in train_loader_v2:
        sequences = sequences.to(device)
        labels    = labels.to(device)

        optimizer_v2.zero_grad()
        outputs = model_lstm_v2(sequences)
        loss    = criterion_v2(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_lstm_v2.parameters(), max_norm=1.0
        )

        optimizer_v2.step()

        total_loss += loss.item()
        _, predicted = outputs.max(dim=1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    avg_loss  = total_loss / len(train_loader_v2)
    train_acc = 100.0 * correct / total

    history_v2['train_loss'].append(avg_loss)
    history_v2['train_acc'].append(train_acc)

    if epoch == 1:
        status = "← starting point"
    elif avg_loss < history_v2['train_loss'][-2]:
        status = "↓ loss improving"
    else:
        status = "→ plateauing"

    print(f"{epoch:<8} {avg_loss:<12.4f} {train_acc:<11.2f}% {status}")

print("-" * 50)
print(f"\nFinal loss:     {history_v2['train_loss'][-1]:.4f}")
print(f"Final accuracy: {history_v2['train_acc'][-1]:.2f}%")
print(f"Random baseline: {100/NUM_CLASSES:.2f}%")

Training on: cpu
Training samples: 89,401

=== TRAINING LSTM v2 (Prefix Sequences) ===
Epoch    Loss         Accuracy     Status
--------------------------------------------------
1        1.9527       38.76      % ← starting point
2        1.4154       56.90      % ↓ loss improving
3        1.2382       61.00      % ↓ loss improving
4        1.1042       65.32      % ↓ loss improving
5        0.9789       69.09      % ↓ loss improving
6        0.9159       70.94      % ↓ loss improving
7        0.8592       72.54      % ↓ loss improving
8        0.8168       74.04      % ↓ loss improving
9        0.7726       75.56      % ↓ loss improving
10       0.7763       76.02      % → plateauing
11       0.7243       77.53      % ↓ loss improving
12       0.7125       78.08      % ↓ loss improving
13       0.6928       78.81      % ↓ loss improving
14       0.6745       79.19      % ↓ loss improving
15       0.6658       79.39      % ↓ loss improving
--------------------------------------------

In [20]:
# Cell 20: Evaluate LSTM v2 on test set
#
# Now predicting next activity at every step
# Expect lower raw accuracy than before (was 95% due to Payment Handled)
# But F1 Macro will be much higher — model actually learns all activities

model_lstm_v2.eval()
all_preds_v2  = []
all_labels_v2 = []

with torch.no_grad():
    for sequences, labels in test_loader_v2:
        sequences = sequences.to(device)
        outputs   = model_lstm_v2(sequences)
        _, predicted = outputs.max(dim=1)
        all_preds_v2.extend(predicted.cpu().numpy())
        all_labels_v2.extend(labels.numpy())

all_preds_v2  = np.array(all_preds_v2)
all_labels_v2 = np.array(all_labels_v2)

# Metrics
accuracy_v2 = accuracy_score(all_labels_v2, all_preds_v2) * 100
f1_w_v2     = f1_score(all_labels_v2, all_preds_v2,
                        average='weighted', zero_division=0)
f1_macro_v2 = f1_score(all_labels_v2, all_preds_v2,
                        average='macro',    zero_division=0)

print(f"=== LSTM v2 RESULTS (Prefix Sequences) ===\n")
print(f"Test Accuracy:        {accuracy_v2:.2f}%")
print(f"F1 Weighted:          {f1_w_v2:.4f}")
print(f"F1 Macro:             {f1_macro_v2:.4f}   ← honest metric")
print(f"Random baseline:      {100/NUM_CLASSES:.2f}%")

# Compare against v1
print(f"\n=== COMPARISON: v1 vs v2 ===")
print(f"{'Metric':<20} {'v1 (final only)':<20} {'v2 (prefix)':<20}")
print("-" * 60)
print(f"{'Accuracy':<20} {'95.19%':<20} {accuracy_v2:.2f}%")
print(f"{'F1 Weighted':<20} {'0.9365':<20} {f1_w_v2:.4f}")
print(f"{'F1 Macro':<20} {'0.2677':<20} {f1_macro_v2:.4f}")
print(f"{'Classes predicted':<20} {'3 / 37':<20}", end=" ")

unique_v2 = len(np.unique(all_preds_v2))
print(f"{unique_v2} / {NUM_CLASSES}")

# Prediction distribution
print(f"\n=== PREDICTION DISTRIBUTION ===")
unique_preds_v2, counts_v2 = np.unique(all_preds_v2, return_counts=True)
print(f"Unique activities predicted: {len(unique_preds_v2)} out of {NUM_CLASSES}")
top5_idx_v2 = np.argsort(counts_v2)[::-1][:5]
print(f"\nTop 5 predicted activities:")
for idx in top5_idx_v2:
    name = encoders['concept:name'].classes_[unique_preds_v2[idx]]
    print(f"  {counts_v2[idx]:5d} times → {name}")

# Per-class report
print(f"\n=== PER-CLASS PERFORMANCE ===")
present_v2   = np.unique(all_labels_v2)
names_v2     = [encoders['concept:name'].classes_[i] for i in present_v2]
print(classification_report(
    all_labels_v2, all_preds_v2,
    labels       = present_v2,
    target_names = names_v2,
    zero_division = 0
))

=== LSTM v2 RESULTS (Prefix Sequences) ===

Test Accuracy:        81.22%
F1 Weighted:          0.7748
F1 Macro:             0.2890   ← honest metric
Random baseline:      2.70%

=== COMPARISON: v1 vs v2 ===
Metric               v1 (final only)      v2 (prefix)         
------------------------------------------------------------
Accuracy             95.19%               81.22%
F1 Weighted          0.9365               0.7748
F1 Macro             0.2677               0.2890
Classes predicted    3 / 37               14 / 37

=== PREDICTION DISTRIBUTION ===
Unique activities predicted: 14 out of 37

Top 5 predicted activities:
   3734 times → Declaration FINAL_APPROVED by SUPERVISOR
   3689 times → Declaration APPROVED by ADMINISTRATION
   3305 times → Request Payment
   3277 times → Payment Handled
   1977 times → Declaration SUBMITTED by EMPLOYEE

=== PER-CLASS PERFORMANCE ===
                                            precision    recall  f1-score   support

    Declaration APPROVED b